# Cohort Attrition Audit — One Operation Per Cell

This notebook reproduces the current filtering pipeline, but **each transformation or filter is isolated in its own cell**.

The goal is to see exactly where we lose:

- rows,
- unique patients,
- studies.

A patient is identified by the pair `STUDY_ID + patient_id`.

**Important:** transformation cells (for example broadcasting age or casting a dtype) do not intentionally remove observations. Filtering cells are reported separately so that losses can be attributed to one rule at a time.


## 1. Imports


In [ ]:
import pandas as pd
from pathlib import Path
from collaborative_care_analysis.config import ENRICHED_DATASET_DIR


## 2. Data path


In [ ]:
ENRICHED_PATH = ENRICHED_DATASET_DIR / "enriched_dataset.csv"


## 3. Reporting helper

This cell only defines functions used to count rows, patients, and studies and to compare a dataset before and after one operation.


In [ ]:
attrition_log = []


def n_patients(df):
    return (
        df[["STUDY_ID", "patient_id"]]
        .drop_duplicates()
        .shape[0]
    )


def n_studies(df):
    return df["STUDY_ID"].nunique()


def report_state(label, df):
    print(label)
    print(f"Rows:     {len(df):,}")
    print(f"Patients: {n_patients(df):,}")
    print(f"Studies:  {n_studies(df):,}")


def report_filter(step, before, after):
    before_studies = set(before["STUDY_ID"].dropna().unique())
    after_studies = set(after["STUDY_ID"].dropna().unique())

    before_ids = set(
        map(
            tuple,
            before[["STUDY_ID", "patient_id"]]
            .drop_duplicates()
            .to_numpy(),
        )
    )
    after_ids = set(
        map(
            tuple,
            after[["STUDY_ID", "patient_id"]]
            .drop_duplicates()
            .to_numpy(),
        )
    )

    dropped_studies = sorted(before_studies - after_studies)
    dropped_patients = before_ids - after_ids

    row_loss = len(before) - len(after)
    patient_loss = len(dropped_patients)
    study_loss = len(dropped_studies)

    attrition_log.append(
        {
            "step": step,
            "rows_before": len(before),
            "rows_after": len(after),
            "rows_lost": row_loss,
            "patients_before": len(before_ids),
            "patients_after": len(after_ids),
            "patients_lost": patient_loss,
            "studies_before": len(before_studies),
            "studies_after": len(after_studies),
            "studies_lost": study_loss,
            "dropped_studies": dropped_studies,
        }
    )

    print(step)
    print(
        f"Rows:     {len(before):,} -> {len(after):,} "
        f"(lost {row_loss:,})"
    )
    print(
        f"Patients: {len(before_ids):,} -> {len(after_ids):,} "
        f"(lost {patient_loss:,})"
    )
    print(
        f"Studies:  {len(before_studies):,} -> {len(after_studies):,} "
        f"(lost {study_loss:,})"
    )

    if dropped_studies:
        print("Dropped studies:")
        for study in dropped_studies:
            print("  -", study)
    else:
        print("Dropped studies: none")


## 4. Load the enriched dataset


In [ ]:
df0 = pd.read_csv(
    ENRICHED_PATH,
    low_memory=False,
)


## 5. Starting dataset size


In [ ]:
report_state("Original dataset", df0)


## 6. Filter rows to integer-valued PHQ-9

**Only operation in this cell:** remove rows where `phq9_total` is missing or not integer-valued.

This reproduces the filtering part of:

```python
df[df["phq9_total"].mod(1).eq(0)]
```


In [ ]:
before = df0

df1 = df0.loc[
    df0["phq9_total"].mod(1).eq(0)
].copy()

report_filter(
    "Keep integer-valued PHQ-9 rows",
    before,
    df1,
)


## 7. Convert PHQ-9 dtype to integer

**Only operation in this cell:** cast `phq9_total` to `int32`.

This is a transformation only; it should not remove any rows, patients, or studies.


In [ ]:
df2 = df1.astype(
    {"phq9_total": "int32"}
)

report_state(
    "After PHQ-9 dtype conversion",
    df2,
)


## 8. Keep only studies that have at least one PHQ-9 value

**Only operation in this cell:** study-level PHQ-9 availability filter.


In [ ]:
before = df2

df3 = (
    df2.groupby("STUDY_ID")
    .filter(
        lambda g: g["phq9_total"].notna().any()
    )
    .copy()
)

report_filter(
    "Keep studies with at least one PHQ-9 value",
    before,
    df3,
)


## 9. Identify patients with a baseline PHQ-9 row

**Only operation in this cell:** create the list of patient IDs that have `follow_up_months == 0.0`.

No filtering happens yet.


In [ ]:
baseline_ids = (
    df3.loc[
        df3["follow_up_months"] == 0.0,
        ["STUDY_ID", "patient_id"],
    ]
    .drop_duplicates()
)

print("Patients with a baseline row:", len(baseline_ids))


## 10. Filter to patients with a baseline PHQ-9 row

**Only operation in this cell:** remove patients who have no baseline row.


In [ ]:
before = df3

df4 = (
    df3.merge(
        baseline_ids.assign(_keep=True),
        on=["STUDY_ID", "patient_id"],
        how="left",
    )
    .loc[lambda x: x["_keep"].notna()]
    .drop(columns="_keep")
    .copy()
)

report_filter(
    "Keep patients with baseline PHQ-9",
    before,
    df4,
)


## 11. Identify patients with a 12-month-window PHQ-9 row

**Only operation in this cell:** create the list of patient IDs with at least one row in the interval 11.5–12.5 months.

No filtering happens yet.


In [ ]:
followup_12m_ids = (
    df4.loc[
        df4["follow_up_months"].between(
            11.5,
            12.5,
            inclusive="both",
        ),
        ["STUDY_ID", "patient_id"],
    ]
    .drop_duplicates()
)

print(
    "Patients with a row in the 11.5-12.5 month window:",
    len(followup_12m_ids),
)


## 12. Filter to patients with a 12-month-window PHQ-9 row

**Only operation in this cell:** remove patients who do not have a row in the 11.5–12.5 month window.


In [ ]:
before = df4

df5 = (
    df4.merge(
        followup_12m_ids.assign(_keep=True),
        on=["STUDY_ID", "patient_id"],
        how="left",
    )
    .loc[lambda x: x["_keep"].notna()]
    .drop(columns="_keep")
    .copy()
)

report_filter(
    "Keep patients with 11.5-12.5 month PHQ-9",
    before,
    df5,
)


## 13. Restrict rows to baseline and the 12-month window

**Only operation in this cell:** remove other follow-up rows.

No patients should be intentionally selected here; the patient-level eligibility was already established above.


In [ ]:
before = df5

df6 = (
    df5.loc[
        (df5["follow_up_months"] == 0.0)
        | df5["follow_up_months"].between(
            11.5,
            12.5,
            inclusive="both",
        )
    ]
    .copy()
)

report_filter(
    "Restrict rows to baseline + 11.5-12.5 months",
    before,
    df6,
)


## 14. Broadcast age within patient

**Only operation in this cell:** forward/back-fill age within each patient.

No filtering.


In [ ]:
df7 = df6.copy()

df7["age"] = (
    df7.groupby(
        ["STUDY_ID", "patient_id"]
    )["age"]
    .transform(
        lambda s: s.ffill().bfill()
    )
    .infer_objects(copy=False)
)

report_state(
    "After broadcasting age",
    df7,
)


## 15. Identify patients with any non-missing age

**Only operation in this cell:** create the eligible patient-ID list.

No filtering yet.


In [ ]:
age_ids = (
    df7.loc[
        df7["age"].notna(),
        ["STUDY_ID", "patient_id"],
    ]
    .drop_duplicates()
)

print("Patients with age:", len(age_ids))


## 16. Filter patients with missing age at all retained timepoints

**Only operation in this cell:** remove patients with no usable age.


In [ ]:
before = df7

df8 = (
    df7.merge(
        age_ids.assign(_keep=True),
        on=["STUDY_ID", "patient_id"],
        how="left",
    )
    .loc[lambda x: x["_keep"].notna()]
    .drop(columns="_keep")
    .copy()
)

report_filter(
    "Keep patients with age",
    before,
    df8,
)


## 17. Broadcast sex within patient

**Only operation in this cell:** forward/back-fill sex within each patient.

No filtering.


In [ ]:
df9 = df8.copy()

df9["sex"] = (
    df9.groupby(
        ["STUDY_ID", "patient_id"]
    )["sex"]
    .transform(
        lambda s: s.ffill().bfill()
    )
    .infer_objects(copy=False)
)

report_state(
    "After broadcasting sex",
    df9,
)


## 18. Identify patients with any non-missing sex

**Only operation in this cell:** create the eligible patient-ID list.

No filtering yet.


In [ ]:
sex_ids = (
    df9.loc[
        df9["sex"].notna(),
        ["STUDY_ID", "patient_id"],
    ]
    .drop_duplicates()
)

print("Patients with sex:", len(sex_ids))


## 19. Filter patients with missing sex at all retained timepoints

**Only operation in this cell:** remove patients with no usable sex.


In [ ]:
before = df9

df10 = (
    df9.merge(
        sex_ids.assign(_keep=True),
        on=["STUDY_ID", "patient_id"],
        how="left",
    )
    .loc[lambda x: x["_keep"].notna()]
    .drop(columns="_keep")
    .copy()
)

report_filter(
    "Keep patients with sex",
    before,
    df10,
)


## 20. Inspect sex labels

**Only operation in this cell:** inspect the labels before casting.

No filtering.


In [ ]:
print(
    "Sex labels:",
    df10["sex"].unique(),
)


## 21. Convert sex to categorical dtype

**Only operation in this cell:** dtype conversion.

No filtering.


In [ ]:
df11 = df10.copy()
df11["sex"] = df11["sex"].astype("category")

report_state(
    "After converting sex to category",
    df11,
)


## 22. Broadcast study arm within patient

**Only operation in this cell:** forward/back-fill `study_arm` within each patient.

No filtering.


In [ ]:
df12 = df11.copy()

df12["study_arm"] = (
    df12.groupby(
        ["STUDY_ID", "patient_id"]
    )["study_arm"]
    .transform(
        lambda s: s.ffill().bfill()
    )
    .infer_objects(copy=False)
)

report_state(
    "After broadcasting study arm",
    df12,
)


## 23. Identify patients with missing study arm

**Only operation in this cell:** create the patient-ID list to remove.

No filtering yet.


In [ ]:
missing_study_arm_ids = (
    df12.loc[
        df12["study_arm"].isna(),
        ["STUDY_ID", "patient_id"],
    ]
    .drop_duplicates()
)

print(
    "Patients with missing study arm:",
    len(missing_study_arm_ids),
)


## 24. Remove patients with missing study arm

**Only operation in this cell:** remove the identified patients.


In [ ]:
before = df12

df13 = (
    df12.merge(
        missing_study_arm_ids.assign(_drop=True),
        on=["STUDY_ID", "patient_id"],
        how="left",
    )
    .loc[lambda x: x["_drop"].isna()]
    .drop(columns="_drop")
    .copy()
)

report_filter(
    "Remove patients with missing study arm",
    before,
    df13,
)


## 25. Identify patients with missing baseline GAD-7

**Only operation in this cell:** identify patients whose baseline row has missing `gad7_total`.

This is completely separate from the baseline PHQ-9 eligibility filter above.


In [ ]:
missing_baseline_gad7_ids = (
    df13.loc[
        df13["gad7_total"].isna()
        & (df13["follow_up_months"] == 0.0),
        ["STUDY_ID", "patient_id"],
    ]
    .drop_duplicates()
)

print(
    "Patients with missing baseline GAD-7:",
    len(missing_baseline_gad7_ids),
)


## 26. Remove patients with missing baseline GAD-7

**Only operation in this cell:** remove the patients identified above.


In [ ]:
before = df13

df14 = (
    df13.merge(
        missing_baseline_gad7_ids.assign(_drop=True),
        on=["STUDY_ID", "patient_id"],
        how="left",
    )
    .loc[lambda x: x["_drop"].isna()]
    .drop(columns="_drop")
    .copy()
)

report_filter(
    "Remove patients with missing baseline GAD-7",
    before,
    df14,
)


## 27. Attrition table — filtering operations only

This table contains only the cells that actually filter rows/patients/studies. Transformations such as broadcasting and dtype conversion are intentionally excluded from the attrition log.


In [ ]:
attrition_df = pd.DataFrame(attrition_log)

attrition_df[
    [
        "step",
        "rows_before",
        "rows_after",
        "rows_lost",
        "patients_before",
        "patients_after",
        "patients_lost",
        "studies_before",
        "studies_after",
        "studies_lost",
        "dropped_studies",
    ]
]


## 28. Save the dataset after every filtering step

This dictionary gives a named snapshot after each actual filter. It is useful for comparing exactly where a study disappears.


In [ ]:
filter_snapshots = {
    "00_original": df0,
    "01_integer_phq9_rows": df1,
    "02_studies_with_phq9": df3,
    "03_baseline_phq9": df4,
    "04_12m_phq9": df5,
    "05_baseline_and_12m_rows_only": df6,
    "06_age": df8,
    "07_sex": df10,
    "08_study_arm": df13,
    "09_baseline_gad7": df14,
}


## 29. Study count after each filtering step


In [ ]:
study_counts = pd.Series(
    {
        step: df["STUDY_ID"].nunique()
        for step, df in filter_snapshots.items()
    },
    name="n_studies",
)

study_counts


## 30. Patient count after each filtering step


In [ ]:
patient_counts = pd.Series(
    {
        step: (
            df[["STUDY_ID", "patient_id"]]
            .drop_duplicates()
            .shape[0]
        )
        for step, df in filter_snapshots.items()
    },
    name="n_patients",
)

patient_counts


## 31. Patients per study after each filtering step

This shows studies that remain present but become very small.


In [ ]:
patients_per_study = {}

for step, step_df in filter_snapshots.items():
    patients_per_study[step] = (
        step_df[
            ["STUDY_ID", "patient_id"]
        ]
        .drop_duplicates()
        .groupby("STUDY_ID")
        .size()
    )

patients_per_study = (
    pd.DataFrame(patients_per_study)
    .fillna(0)
    .astype(int)
)

patients_per_study


## 32. Study survival matrix

`True` means the study is still present after that filtering step.


In [ ]:
all_studies = sorted(
    set().union(
        *[
            set(df["STUDY_ID"].dropna().unique())
            for df in filter_snapshots.values()
        ]
    )
)

study_survival = pd.DataFrame(
    index=all_studies
)

for step, step_df in filter_snapshots.items():
    present = set(
        step_df["STUDY_ID"]
        .dropna()
        .unique()
    )
    study_survival[step] = [
        study in present
        for study in all_studies
    ]

study_survival


## 33. Final filtered dataset

`df14` is the endpoint after applying the same sequence of filters, with every operation made explicit.


In [ ]:
filtered_df = df14.copy()

report_state(
    "Final filtered dataset",
    filtered_df,
)


## 34. The 13 studies and the 9 studies

Prints the 13 studies still present before the baseline GAD-7 filter (`df13`) and the 9 studies that remain in the final dataset (`df14`), with patient counts.

In [ ]:
def patients_by_study(df):
    return (
        df[["STUDY_ID", "patient_id"]]
        .drop_duplicates()
        .groupby("STUDY_ID")
        .size()
    )


studies_13 = patients_by_study(df13)
studies_9 = patients_by_study(df14)

print(f"The {len(studies_13)} studies (before baseline GAD-7 filter):")
for study, n in studies_13.items():
    status = "kept" if study in studies_9.index else "dropped by GAD-7 filter"
    print(f"  - {study:<22} {n:>5,} patients  ({status})")

print()
print(f"The {len(studies_9)} studies (final dataset):")
for study, n in studies_9.items():
    print(f"  - {study:<22} {n:>5,} patients")